# Cell 0 — Stage 7: Canal Blockage Detection

**Goal:** Detect blockages (vegetation, debris, silt) within the canal mask from Stage 5.

| Cell | Purpose | Time |
|------|---------|------|
| 0 | Header | - |
| 1 | Config & imports | ~2s |
| 2 | Load canal model & build full-mosaic canal mask | ~30-60 min |
| 2b | Fast reload saved masks + relaxed filters | ~30s |
| 3 | Extract canal centerlines & segment into reaches | ~2 min |
| 4 | Classify each reach: clear vs blocked | ~5 min |
| 5 | Full-mosaic blockage map visualization | ~1 min |
| 6 | Summary statistics & export GeoTIFF | ~2 min |
| 7 | Detailed zoomed patches | ~1 min |

**Input:** `best_binary_unet_v5.pth` from Stage 5 + orthomosaic TIFs  
**Output:** Blockage map (GeoTIFF), summary report

In [2]:
# Cell 1 — Config & imports
import os, time, warnings, json, gc
warnings.filterwarnings('ignore')

import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_bounds
import cv2
from scipy import ndimage
from skimage.morphology import skeletonize

import torch
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DATA_DIR = r'c:/Users/PRABHAKAR/Documents/Wells-Lab'
OUT_DIR  = os.path.join(DATA_DIR, 'New', 'results')
OUT_05b  = os.path.join(DATA_DIR, 'outputs', '05b_binary')
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PATCH_SIZE = 512
THRESHOLD  = 0.55

# New-mission orthos. NRBC = true RGB (4-band uint8). MISSION_3_MS = converted
# false-color RGB (no blue band) -> EXPERIMENTAL, results not trustworthy.
NEW_DIR = os.path.join(DATA_DIR, 'New')
ALL_FILES = {
    'NRBC_MISSION_1': (os.path.join(NEW_DIR, 'NRBC_D10_MISSION_1_ortho.tif'),          0.0339),
    'NRBC_SURVEY_1':  (os.path.join(NEW_DIR, 'NRBC_D10_Survey_1_ortho.tif'),           0.0343),
}

IMG_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMG_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

MODEL_NAME = 'best_binary_unet_v5.pth'

print(f'Device: {DEVICE}')
_mp = os.path.join(OUT_05b, MODEL_NAME)
print(f'Canal model: {_mp}  [{"OK" if os.path.exists(_mp) else "MISSING"}]')
print('Orthomosaic files:')
for _tag, (_fp, _px) in ALL_FILES.items():
    print(f'  {_tag:11s} [{"OK     " if os.path.exists(_fp) else "MISSING"}]  {_fp}')

Device: cuda
Orthomosaics: ['ATTANUR_1', 'ATTANUR_2', 'ATTANUR_3', 'ATTANUR_4', 'SHAKAPUR']
Canal model: c:/Users/PRABHAKAR/Documents/Wells-Lab\outputs\05b_binary\best_binary_unet_v5.pth


In [3]:
# Cell 2 — Load canal model + TTA helper
# SHAKAPUR excluded (test set). TTA=4 rotations, STEP=384, DS=8

gc.collect()
torch.cuda.empty_cache()

model_path = os.path.join(OUT_05b, MODEL_NAME)
model = smp.Unet(encoder_name='resnet50', encoder_weights=None,
                  in_channels=3, classes=1).to(DEVICE)
model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
model.eval()
print(f'Loaded: {model_path}')

# Remove SHAKAPUR — it's for testing only
PROCESS_FILES = {k: v for k, v in ALL_FILES.items() if k != 'SHAKAPUR'}

TILE = 512
STEP = 384
DS = 8
MAX_WIDTH_M = 15.0
MIN_AREA_PX = 100
MIN_ASPECT = 2.0
MIN_SKEL_RATIO = 0.015

def tta_predict_4(model, img_t):
    preds = []
    for k in range(4):
        x = torch.rot90(img_t, k, dims=[2, 3])
        with torch.amp.autocast('cuda'):
            prob = torch.sigmoid(model(x))
        prob = torch.rot90(prob, -k, dims=[2, 3])
        preds.append(prob)
    return torch.stack(preds).mean(dim=0)



Loaded: c:/Users/PRABHAKAR/Documents/Wells-Lab\outputs\05b_binary\best_binary_unet_v5.pth


In [4]:
# Cell 3 — build_canal_mask() — tiled inference + component filters
def build_canal_mask(tag, fpath, model, px_m, tile=TILE, step=STEP, ds=DS):
    with rasterio.open(fpath) as src:
        W, H = src.width, src.height
        crs = src.crs
        transform = src.transform

    out_h, out_w = H // ds, W // ds
    tile_ds = tile // ds
    pred_sum = np.zeros((out_h, out_w), dtype=np.float32)
    count_map = np.zeros((out_h, out_w), dtype=np.float32)

    total = ((H - tile) // step + 1) * ((W - tile) // step + 1)
    done = 0
    t0 = time.time()

    with rasterio.open(fpath) as src:
        for row in range(0, H - tile + 1, step):
            for col in range(0, W - tile + 1, step):
                data = src.read([1, 2, 3, 4],
                                window=Window(col, row, tile, tile))
                valid = data[3] == 255
                if valid.sum() < tile * tile * 0.3:
                    done += 1
                    continue

                rgb = data[:3].astype(np.float32) / 255.0
                img_t = torch.from_numpy(rgb).float()
                img_t = (img_t - IMG_MEAN) / IMG_STD

                with torch.no_grad():
                    prob = tta_predict_4(model, img_t.unsqueeze(0).to(DEVICE))
                    prob = prob.cpu().numpy()[0, 0].astype(np.float32)

                prob[~valid] = 0
                valid_f = valid.astype(np.float32)

                prob_ds = prob.reshape(tile_ds, ds, tile_ds, ds).mean(axis=(1, 3))
                valid_ds = valid_f.reshape(tile_ds, ds, tile_ds, ds).mean(axis=(1, 3))

                r0 = row // ds
                c0 = col // ds
                r1 = min(r0 + tile_ds, out_h)
                c1 = min(c0 + tile_ds, out_w)

                pred_sum[r0:r1, c0:c1] += prob_ds[:r1-r0, :c1-c0]
                count_map[r0:r1, c0:c1] += valid_ds[:r1-r0, :c1-c0]
                done += 1

            if done % 500 == 0:
                elapsed = time.time() - t0
                rate = done / max(1, elapsed)
                eta = (total - done) / max(1, rate)
                print(f'    {done}/{total} tiles ... ({elapsed:.0f}s, ETA {eta/60:.0f}min)')

    # memory-frugal averaging (huge tiles): in-place divide, prob as float16
    count_map[count_map == 0] = 1.0
    pred_sum /= count_map
    del count_map; gc.collect()
    canal_raw = (pred_sum > THRESHOLD).astype(np.uint8)
    pred_avg = pred_sum.astype(np.float16)
    del pred_sum; gc.collect()

    px_m_ds = px_m * ds
    print(f'  Raw canal pixels: {canal_raw.sum()} ({canal_raw.sum()/max(1,canal_raw.size)*100:.2f}%)')

    # 1. Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    canal_clean = cv2.morphologyEx(canal_raw, cv2.MORPH_OPEN, kernel, iterations=1)
    canal_clean = cv2.morphologyEx(canal_clean, cv2.MORPH_CLOSE, kernel, iterations=1)

    # 2. Connected component analysis
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(canal_clean, connectivity=8)
    canal_filtered = np.zeros_like(canal_clean)
    kept = 0
    removed_reasons = {'small': 0, 'blobby': 0, 'too_wide': 0, 'no_edge': 0, 'fat_blob': 0}

    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        x, y, w, h = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP], \
                      stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]

        # Filter 1: Remove small blobs
        if area < MIN_AREA_PX:
            removed_reasons['small'] += 1
            continue

        # Filter 2: Aspect ratio — real canals are elongated
        bbox_aspect = max(w, h) / max(1, min(w, h))
        if bbox_aspect < MIN_ASPECT:
            removed_reasons['blobby'] += 1
            continue

        # Filter 3: Width check via distance transform
        comp_mask = (labels[y:y+h, x:x+w] == i).astype(np.uint8)
        if comp_mask.sum() > 0:
            dt = cv2.distanceTransform(comp_mask, cv2.DIST_L2, 5)
            max_width_px = dt.max() * 2
            max_width_m = max_width_px * px_m_ds
            if max_width_m > MAX_WIDTH_M:
                removed_reasons['too_wide'] += 1
                continue

        # Filter 4: Skeleton/area ratio — canals are thin & long, blobs are fat
        skel = skeletonize(comp_mask > 0).astype(np.uint8)
        skel_len = skel.sum()
        skel_ratio = skel_len / max(1, area)
        if skel_ratio < MIN_SKEL_RATIO:
            removed_reasons['fat_blob'] += 1
            continue

        # Filter 5: Edge connectivity — only remove small isolated blobs
        comp_full = (labels == i)
        touches_top = comp_full[0, :].any()
        touches_bottom = comp_full[-1, :].any()
        touches_left = comp_full[:, 0].any()
        touches_right = comp_full[:, -1].any()

        margin = 5
        touches_valid_edge = (y < margin or (y + h) > (out_h - margin) or
                              x < margin or (x + w) > (out_w - margin))
        touches_any_edge = touches_top or touches_bottom or touches_left or touches_right or touches_valid_edge

        if not touches_any_edge and area < 2000:
            removed_reasons['no_edge'] += 1
            continue

        canal_filtered[labels == i] = 1
        kept += 1

    print(f'  Components: {n_labels-1} total, {kept} kept')
    print(f'  Removed: {removed_reasons}')
    print(f'  Filtered canal pixels: {canal_filtered.sum()} ({canal_filtered.sum()/max(1,canal_filtered.size)*100:.2f}%)')

    new_transform = rasterio.transform.Affine(
        transform.a * ds, transform.b, transform.c,
        transform.d, transform.e * ds, transform.f
    )
    geo_info = {'crs': crs, 'transform': new_transform,
                'width': out_w, 'height': out_h, 'orig_W': W, 'orig_H': H}

    dt_elapsed = time.time() - t0
    print(f'  {tag}: {out_w}x{out_h}, {dt_elapsed/60:.1f} min')
    return canal_filtered, pred_avg, geo_info



In [5]:
# Cell 4 — Run inference: build & save canal masks
# Build canal masks for 4 mosaics (SHAKAPUR excluded)
canal_data = {}
for tag, (fpath, px_m) in PROCESS_FILES.items():
    flag = os.path.join(OUT_DIR, f'_done_{tag}.flag')
    if os.path.exists(flag):
        print(f'{tag}: already inferred this run -- skipping (resume)')
        continue
    print(f'\nProcessing {tag} ...')
    canal_bin, canal_prob, geo = build_canal_mask(tag, fpath, model, px_m)
    canal_data[tag] = {
        'binary': canal_bin, 'prob': canal_prob, 'geo': geo,
        'fpath': fpath, 'px_m': px_m
    }
    gc.collect()
    torch.cuda.empty_cache()

    np.savez_compressed(os.path.join(OUT_DIR, f'{tag}_canal_mask.npz'),
                        binary=canal_bin, prob=canal_prob)
    open(flag, 'w').close()   # mark mosaic done (crash-resume)

del model
gc.collect()
torch.cuda.empty_cache()
print(f'\nDone — canal masks for {len(canal_data)} mosaics (SHAKAPUR excluded for testing)')



Processing ATTANUR_1 ...


RasterioIOError: c:/Users/PRABHAKAR/Documents/Wells-Lab\TLBC_D95_ATTANUR_1_ortho.tif: No such file or directory

In [ ]:
# Cell 5 — Canal-vs-FIC filter parameters
# NO retraining needed — reloads saved .npz from Cell 2 and creates clean masks

gc.collect()
torch.cuda.empty_cache()

PROCESS_FILES = {k: v for k, v in ALL_FILES.items() if k != 'SHAKAPUR'}
DS = 8

# Component-level filters at DS=8 mask resolution.
# FICs are usually narrow/short; real canals/distributaries are wider, longer, and less compact.
MIN_COMPONENT_AREA_PX = 250
MIN_CANAL_ASPECT = 3.0
MIN_SKEL_RATIO = 0.008
MIN_CANAL_MEAN_WIDTH_M = 2.0
MIN_CANAL_MAX_WIDTH_M = 3.0
MAX_CANAL_WIDTH_M = 35.0
# --- Tuned for new farmland areas: drop blocky fields, keep thin+long canals ---
MAX_CANAL_MEAN_WIDTH_M = 6.0     # avg width above this = a field/block, not a canal
MIN_CANAL_LENGTH_M     = 180.0    # drop short field-edge fragments
MIN_LEN_WIDTH_RATIO    = 22.0    # canals are very elongated (length >> width)
KEEP_LONGEST           = 1       # keep only the longest components per tile



In [ ]:
# Cell 6 — classify_canal_components() function  (tuned: thin+long+elongated, keep-longest)
def classify_canal_components(canal_raw, px_m_ds):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    canal_clean = cv2.morphologyEx(canal_raw, cv2.MORPH_OPEN, kernel, iterations=1)
    canal_clean = cv2.morphologyEx(canal_clean, cv2.MORPH_CLOSE, kernel, iterations=2)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(canal_clean, connectivity=8)
    canal_only = np.zeros_like(canal_clean, dtype=np.uint8)
    fic_candidate = np.zeros_like(canal_clean, dtype=np.uint8)
    rejected = np.zeros_like(canal_clean, dtype=np.uint8)
    counts = {'canal': 0, 'fic_candidate': 0, 'small': 0, 'compact': 0,
              'too_wide': 0, 'fat_blob': 0}

    cand = []   # (label_id, length_m) of canal-like components, for keep-longest
    for i in range(1, n_labels):
        area = int(stats[i, cv2.CC_STAT_AREA])
        x, y, w, h = [int(stats[i, k]) for k in (cv2.CC_STAT_LEFT, cv2.CC_STAT_TOP,
                                                  cv2.CC_STAT_WIDTH, cv2.CC_STAT_HEIGHT)]
        comp = (labels[y:y+h, x:x+w] == i).astype(np.uint8)
        full = labels == i

        if area < MIN_COMPONENT_AREA_PX:
            rejected[full] = 1; counts['small'] += 1; continue
        aspect = max(w, h) / max(1, min(w, h))
        if aspect < MIN_CANAL_ASPECT:
            rejected[full] = 1; counts['compact'] += 1; continue

        dt = cv2.distanceTransform(comp, cv2.DIST_L2, 5)
        width_vals = dt[comp > 0] * 2.0 * px_m_ds
        mean_width_m = float(np.mean(width_vals)) if width_vals.size else 0.0
        max_width_m = float(np.max(width_vals)) if width_vals.size else 0.0

        if max_width_m > MAX_CANAL_WIDTH_M:
            rejected[full] = 1; counts['too_wide'] += 1; continue
        if mean_width_m > MAX_CANAL_MEAN_WIDTH_M:        # too wide on average = field/blob
            rejected[full] = 1; counts['too_wide'] += 1; continue

        skel = skeletonize(comp > 0).astype(np.uint8)
        skel_ratio = float(skel.sum() / max(1, area))
        if skel_ratio < MIN_SKEL_RATIO:
            rejected[full] = 1; counts['fat_blob'] += 1; continue

        length_m = float(skel.sum() * px_m_ds)
        if length_m < MIN_CANAL_LENGTH_M:                # too short
            rejected[full] = 1; counts['small'] += 1; continue
        if length_m / max(mean_width_m, 0.1) < MIN_LEN_WIDTH_RATIO:   # not elongated
            rejected[full] = 1; counts['compact'] += 1; continue

        if mean_width_m < MIN_CANAL_MEAN_WIDTH_M or max_width_m < MIN_CANAL_MAX_WIDTH_M:
            fic_candidate[full] = 1; counts['fic_candidate'] += 1; continue

        cand.append((i, length_m))

    # keep only the longest canal-like components (isolates the main canal line)
    cand.sort(key=lambda t: t[1], reverse=True)
    if KEEP_LONGEST and len(cand) > KEEP_LONGEST:
        cand = cand[:KEEP_LONGEST]
    for i, _ in cand:
        canal_only[labels == i] = 1; counts['canal'] += 1

    return canal_only, fic_candidate, rejected, counts

In [ ]:
# Cell 7 — Reload masks & apply canal-only / FIC filter
canal_data = {}

for tag, (fpath, px_m) in PROCESS_FILES.items():
    print(f'\n{"="*60}')
    print(f'Processing {tag} ...')

    mask_path = os.path.join(OUT_DIR, f'{tag}_canal_mask.npz')
    if not os.path.exists(mask_path):
        print(f'  SKIP — {mask_path} not found. Run Cell 2 first.')
        continue

    saved = np.load(mask_path)
    canal_prob = saved['prob']
    canal_raw = (canal_prob > THRESHOLD).astype(np.uint8)
    out_h, out_w = canal_raw.shape
    px_m_ds = px_m * DS

    canal_only, fic_candidate, rejected, counts = classify_canal_components(canal_raw, px_m_ds)

    print(f'  Loaded: {out_w}x{out_h}')
    print(f'  Raw predicted pixels: {canal_raw.sum()} ({canal_raw.sum()/max(1,canal_raw.size)*100:.2f}%)')
    print(f'  Canal-only pixels:    {canal_only.sum()} ({canal_only.sum()/max(1,canal_only.size)*100:.2f}%)')
    print(f'  FIC-candidate pixels: {fic_candidate.sum()}')
    print(f'  Component counts: {counts}')

    with rasterio.open(fpath) as src:
        crs = src.crs
        transform = src.transform
    new_transform = rasterio.transform.Affine(
        transform.a * DS, transform.b, transform.c,
        transform.d, transform.e * DS, transform.f
    )
    geo_info = {'crs': crs, 'transform': new_transform,
                'width': out_w, 'height': out_h,
                'orig_W': int(out_w * DS), 'orig_H': int(out_h * DS)}

    canal_data[tag] = {
        'binary': canal_only, 'prob': canal_prob, 'raw': canal_raw,
        'fic_candidate': fic_candidate, 'rejected': rejected,
        'filter_counts': counts, 'geo': geo_info,
        'fpath': fpath, 'px_m': px_m
    }

    np.savez_compressed(mask_path, binary=canal_only, prob=canal_prob,
                        raw=canal_raw, fic_candidate=fic_candidate,
                        rejected=rejected)
    clean_path = os.path.join(OUT_DIR, f'{tag}_canal_only_mask.npz')
    np.savez_compressed(clean_path, binary=canal_only, prob=canal_prob,
                        fic_candidate=fic_candidate, rejected=rejected)

    del saved, canal_raw, canal_only, fic_candidate, rejected
    gc.collect()

print(f'\nDone — cleaned canal masks for {len(canal_data)} mosaics')


In [ ]:
# Cell 8 — Extract centerlines, widths, segment into reaches
DS_A = 4  # analysis downscale on top of DS from Cell 2
REACH_LEN_PX = 100

results = {}

for tag, cdata in canal_data.items():
    print(f'\n{"="*60}')
    print(f'Processing {tag} ...')
    canal_bin = cdata['binary']
    px_m = cdata['px_m'] * DS
    h, w = canal_bin.shape

    h_s = h // DS_A
    w_s = w // DS_A
    canal_s = cv2.resize(canal_bin, (w_s, h_s), interpolation=cv2.INTER_NEAREST)
    px_m_a = px_m * DS_A
    print(f'  Full: {w}x{h} -> Analysis: {w_s}x{h_s} ({canal_s.nbytes/1e6:.1f} MB)')

    dist = cv2.distanceTransform(canal_s, cv2.DIST_L2, 5)
    width_arr = dist * 2.0 * px_m_a
    del dist
    gc.collect()

    skel = skeletonize(canal_s > 0).astype(np.uint8)
    print(f'  Centerline pixels: {skel.sum()}')

    n_cc, labels_s = cv2.connectedComponents(canal_s, connectivity=8)
    print(f'  Canal segments: {n_cc - 1}')

    sr, sc = np.where(skel > 0)
    sl = labels_s[sr, sc]
    print(f'  Skeleton pixels: {len(sr)}')
    del skel
    gc.collect()

    reaches = []
    for cid in range(1, n_cc):
        m = sl == cid
        cr = sr[m]
        cc_arr = sc[m]
        if len(cr) < 10:
            continue

        pts = np.column_stack([cr, cc_arr])
        n_r = max(1, len(pts) // REACH_LEN_PX)

        for ri in range(n_r):
            s = ri * REACH_LEN_PX
            e = min((ri + 1) * REACH_LEN_PX, len(pts))
            rp = pts[s:e]

            rmin_s = max(0, rp[:, 0].min() - 10)
            cmin_s = max(0, rp[:, 1].min() - 10)
            rmax_s = min(h_s, rp[:, 0].max() + 10)
            cmax_s = min(w_s, rp[:, 1].max() + 10)

            rc = (labels_s[rmin_s:rmax_s, cmin_s:cmax_s] == cid)
            cpx = int(rc.sum())
            if cpx < 10:
                continue

            wv = width_arr[rmin_s:rmax_s, cmin_s:cmax_s][rc]

            reaches.append({
                'canal_id': int(cid),
                'reach_idx': ri,
                'bbox': (rmin_s * DS_A, cmin_s * DS_A,
                         min(h, rmax_s * DS_A), min(w, cmax_s * DS_A)),
                'bbox_s': (rmin_s, cmin_s, rmax_s, cmax_s),
                'center': (int(rp[:, 0].mean() * DS_A), int(rp[:, 1].mean() * DS_A)),
                'canal_px': cpx,
                'avg_width_m': float(wv.mean()),
                'max_width_m': float(wv.max()),
                'length_m': float(len(rp) * px_m_a),
            })

    lbl_dtype = np.uint8 if n_cc < 256 else np.uint16
    results[tag] = {
        'reaches': reaches,
        'labels_s': labels_s.astype(lbl_dtype),
        'ds_a': DS_A,
        'shape_s': (h_s, w_s),
        'n_canals': n_cc - 1,
    }
    del canal_s, width_arr, labels_s, sr, sc, sl
    gc.collect()

    print(f'  Reaches: {len(reaches)}')
    if reaches:
        print(f'  Avg width: {np.mean([r["avg_width_m"] for r in reaches]):.2f}m')

print(f'\n{"="*60}')
print(f'Total reaches: {sum(len(r["reaches"]) for r in results.values())}')


In [ ]:
# Cell 9 — Reach feature + blocked-length functions
# Uses cleaned canal-only mask from Cell 2b; FIC candidates are excluded before this step.

def compute_reach_features_and_blockage(rgb_patch, canal_mask_patch):
    canal_px = canal_mask_patch > 0
    if canal_px.sum() < 10:
        return None, None

    r = rgb_patch[:, :, 0].astype(np.float32)
    g = rgb_patch[:, :, 1].astype(np.float32)
    b = rgb_patch[:, :, 2].astype(np.float32)
    hsv_patch = cv2.cvtColor(rgb_patch, cv2.COLOR_RGB2HSV)
    h_ch = hsv_patch[:, :, 0].astype(np.float32)
    s_ch = hsv_patch[:, :, 1].astype(np.float32)
    v_ch = hsv_patch[:, :, 2].astype(np.float32)

    exg = 2.0 * g - r - b
    denom = g + r - b + 1e-6
    vari = np.where(np.abs(denom) > 1, (g - r) / denom, 0)
    green_hsv = ((h_ch >= 35) & (h_ch <= 85) & (s_ch > 40) & (v_ch > 50))
    veg_mask = ((exg > 25) | ((vari > 0.1) & green_hsv)) & canal_px
    water_mask = (v_ch < 120) & (b > r) & (s_ch > 15) & canal_px
    bright_mask = (v_ch > 180) & canal_px
    silt_like_mask = bright_mask & (s_ch < 50)
    gray = (0.299 * r + 0.587 * g + 0.114 * b)

    n_px = max(1, int(canal_px.sum()))
    feats = {
        'veg_frac':       float(veg_mask.sum() / n_px),
        'green_hsv_frac': float((green_hsv & canal_px).sum() / n_px),
        'exg_mean':       float(exg[canal_px].mean()),
        'vari_mean':      float(vari[canal_px].mean()),
        'water_frac':     float(water_mask.sum() / n_px),
        'bright_frac':    float(bright_mask.sum() / n_px),
        'silt_like_frac': float(silt_like_mask.sum() / n_px),
        'gray_std':       float(gray[canal_px].std()),
        'v_mean':         float(v_ch[canal_px].mean()),
        's_mean':         float(s_ch[canal_px].mean()),
    }
    return feats, (veg_mask | silt_like_mask).astype(np.uint8)

def centerline_blocked_length(canal_patch, blocked_mask, reach_length_m):
    skel = skeletonize(canal_patch > 0).astype(np.uint8)
    skel_count = int(skel.sum())
    if skel_count == 0:
        return 0.0, 0.0
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    blocked_dil = cv2.dilate(blocked_mask.astype(np.uint8), kernel, iterations=1)
    blocked_skel = int(((skel > 0) & (blocked_dil > 0)).sum())
    blocked_frac = blocked_skel / max(1, skel_count)
    return float(reach_length_m * blocked_frac), float(blocked_frac)



In [ ]:
# Cell 10 — Classify reaches: vegetation / silt scoring
print('Classifying canal-only reaches and measuring blocked length ...\n')

for tag, cdata in canal_data.items():
    print(f'{tag}:')
    fpath = cdata['fpath']
    reaches = results[tag]['reaches']

    with rasterio.open(fpath) as src:
        W, H = src.width, src.height

    with rasterio.open(fpath) as src:
        for reach in reaches:
            r_min, c_min, r_max, c_max = reach['bbox']
            fr_min, fc_min = r_min * DS, c_min * DS
            fr_max, fc_max = min(r_max * DS, H), min(c_max * DS, W)
            rh = fr_max - fr_min
            rw = fc_max - fc_min
            if rh < 8 or rw < 8:
                reach['status'] = 'unknown'; reach['features'] = None
                reach['blocked_distance_m'] = 0.0; reach['blocked_fraction'] = 0.0
                continue

            data = src.read([1, 2, 3], window=Window(fc_min, fr_min, rw, rh))
            rgb_full = data.transpose(1, 2, 0)
            ds_total = DS * results[tag]['ds_a']
            crop_h = (rh // ds_total) * ds_total
            crop_w = (rw // ds_total) * ds_total
            if crop_h < ds_total or crop_w < ds_total:
                reach['status'] = 'unknown'; reach['features'] = None
                reach['blocked_distance_m'] = 0.0; reach['blocked_fraction'] = 0.0
                continue

            rgb_sm = rgb_full[:crop_h, :crop_w].reshape(
                crop_h // ds_total, ds_total,
                crop_w // ds_total, ds_total, 3
            ).mean(axis=(1, 3)).astype(np.uint8)

            rmin_s, cmin_s, rmax_s, cmax_s = reach['bbox_s']
            labels_s = results[tag]['labels_s']
            cid = reach['canal_id']
            canal_patch = (labels_s[rmin_s:rmin_s + rgb_sm.shape[0],
                                    cmin_s:cmin_s + rgb_sm.shape[1]] == cid).astype(np.uint8)
            if canal_patch.shape != rgb_sm.shape[:2]:
                mh = min(canal_patch.shape[0], rgb_sm.shape[0])
                mw = min(canal_patch.shape[1], rgb_sm.shape[1])
                canal_patch = canal_patch[:mh, :mw]
                rgb_sm = rgb_sm[:mh, :mw]

            feats, blocked_mask = compute_reach_features_and_blockage(rgb_sm, canal_patch)
            reach['features'] = feats
            if feats is None:
                reach['status'] = 'unknown'
                reach['blocked_distance_m'] = 0.0; reach['blocked_fraction'] = 0.0
                continue

            blockage_types = []
            score = 0.0
            if feats['veg_frac'] > 0.5:
                score += 4.0; blockage_types.append('heavy_vegetation')
            elif feats['veg_frac'] > 0.25:
                score += 2.5; blockage_types.append('moderate_vegetation')
            elif feats['veg_frac'] > 0.1:
                score += 1.0; blockage_types.append('light_vegetation')

            if feats['silt_like_frac'] > 0.35:
                score += 2.0; blockage_types.append('silt_like_deposit')
            elif feats['silt_like_frac'] > 0.15:
                score += 1.0; blockage_types.append('light_silt_like')

            if feats['gray_std'] > 55 and feats['water_frac'] < 0.1:
                score += 1.0; blockage_types.append('structural_irregularity')
            if feats['water_frac'] > 0.4 and feats['veg_frac'] < 0.1:
                score -= 1.5

            blocked_dist, blocked_frac = centerline_blocked_length(canal_patch, blocked_mask, reach['length_m'])
            reach['blocked_distance_m'] = blocked_dist
            reach['blocked_fraction'] = blocked_frac

            if score >= 3.0 or blocked_frac >= 0.45:
                reach['status'] = 'blocked'
            elif score >= 1.5 or blocked_frac >= 0.15:
                reach['status'] = 'partial'
            else:
                reach['status'] = 'clear'

            reach['blockage_score'] = float(score)
            reach['blockage_types'] = blockage_types

    status_counts = {}
    for r in reaches:
        s = r.get('status', 'unknown')
        status_counts[s] = status_counts.get(s, 0) + 1
    print(f'  Reaches: {len(reaches)} | Results: {status_counts}')

print('\nClassification complete.')


In [ ]:
# Cell 11 — Visualization helper (contrast stretch)
VIS_MAX = 2000
STATUS_COLORS = {
    'clear':   (0, 200, 0),
    'partial': (255, 165, 0),
    'blocked': (255, 0, 0),
    'unknown': (128, 128, 128),
}

def contrast_stretch_b4(rgb, valid_mask):
    out = np.zeros_like(rgb, dtype=np.uint8)
    for c in range(3):
        ch = rgb[:,:,c].astype(np.float64)
        vpx = ch[valid_mask]
        if vpx.size < 100: continue
        lo, hi = np.percentile(vpx, [2, 98])
        if hi - lo < 5: continue
        s = np.clip((ch - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)
        s[~valid_mask] = 0
        out[:,:,c] = s
    return out



In [ ]:
# Cell 12 — Full-mosaic blockage map
n_mosaics = len(canal_data)
fig, axes = plt.subplots(n_mosaics, 2, figsize=(16, 7 * n_mosaics))
if n_mosaics == 1:
    axes = axes[np.newaxis, :]

for idx, (tag, cdata) in enumerate(canal_data.items()):
    fpath = cdata['fpath']
    reaches = results[tag]['reaches']
    labels_s = results[tag]['labels_s']
    h_s, w_s = results[tag]['shape_s']
    ds_a = results[tag]['ds_a']
    canal_bin = cdata['binary']
    h_full, w_full = canal_bin.shape

    vis_scale = max(1, max(h_s, w_s) // VIS_MAX)
    vh, vw = h_s // vis_scale, w_s // vis_scale

    with rasterio.open(fpath) as src:
        thumb = src.read([1, 2, 3, 4], out_shape=(4, vh, vw))
    rgb_vis = thumb[:3].transpose(1, 2, 0)
    valid_vis = thumb[3] == 255
    rgb_vis = contrast_stretch_b4(rgb_vis, valid_vis)
    del thumb

    # Canal overlay from small labels
    canal_s = (labels_s > 0).astype(np.uint8)
    canal_vis = cv2.resize(canal_s, (vw, vh), interpolation=cv2.INTER_NEAREST)
    overlay_canal = rgb_vis.copy()
    overlay_canal[canal_vis == 1] = (overlay_canal[canal_vis == 1] * 0.5 +
                                      np.array([0, 255, 255]) * 0.5).astype(np.uint8)

    # Blockage map at small resolution, then resize to vis
    bmap = np.zeros((h_s, w_s, 3), dtype=np.uint8)
    for reach in reaches:
        status = reach.get('status', 'unknown')
        color = STATUS_COLORS.get(status, (128, 128, 128))
        rmin_s, cmin_s, rmax_s, cmax_s = reach['bbox_s']
        cid = reach['canal_id']
        region = labels_s[rmin_s:rmax_s, cmin_s:cmax_s]
        mask = (region == cid)
        for ci in range(3):
            bmap[rmin_s:rmax_s, cmin_s:cmax_s, ci][mask] = color[ci]

    bmap_vis = cv2.resize(bmap, (vw, vh), interpolation=cv2.INTER_NEAREST)
    has_color = bmap_vis.sum(axis=2) > 0
    blockage_overlay = rgb_vis.copy()
    blockage_overlay[has_color] = (blockage_overlay[has_color] * 0.35 +
                                    bmap_vis[has_color] * 0.65).astype(np.uint8)

    axes[idx, 0].imshow(overlay_canal)
    axes[idx, 0].set_title(f'{tag} — Canal mask (cyan)', fontsize=11)
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(blockage_overlay)
    axes[idx, 1].set_title(f'{tag} — Blockage status', fontsize=11)
    axes[idx, 1].axis('off')

    active = reaches
    sc = {}
    for r in active:
        s = r.get('status', 'unknown')
        sc[s] = sc.get(s, 0) + 1
    stats_str = ', '.join(f'{k}: {v}' for k, v in sorted(sc.items()))
    axes[idx, 1].text(0.02, 0.02, stats_str, transform=axes[idx, 1].transAxes,
                       fontsize=9, color='white', backgroundcolor='black',
                       verticalalignment='bottom')

    del bmap, bmap_vis, blockage_overlay, overlay_canal, rgb_vis, canal_vis
    gc.collect()

legend_patches = [mpatches.Patch(color=np.array(c)/255, label=s.capitalize())
                  for s, c in STATUS_COLORS.items()]
fig.legend(handles=legend_patches, loc='upper center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, 1.02))
plt.suptitle('Canal Blockage Detection — TLBC D95 (FICs excluded)', fontsize=14, y=1.05)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'blockage_map_all.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved to: {os.path.join(OUT_DIR, "blockage_map_all.png")}')


In [ ]:
# Cell 13 — Blockage summary statistics
STATUS_VALUES = {'clear': 1, 'partial': 2, 'blocked': 3, 'unknown': 0}

print('='*60)
print('CANAL BLOCKAGE SUMMARY — TLBC D95')
print('Scope: cleaned canal-only mask; FIC candidates excluded before reach extraction')
print('='*60)

total_reaches = 0
total_clear = 0
total_partial = 0
total_blocked = 0
total_length_clear = 0.0
total_length_partial = 0.0
total_length_blocked = 0.0
all_blockage_types = {}

for tag in canal_data:
    reaches = results[tag]['reaches']
    active = reaches
    n_fic = 0
    n = len(active)
    clear   = sum(1 for r in active if r.get('status') == 'clear')
    partial = sum(1 for r in active if r.get('status') == 'partial')
    blocked = sum(1 for r in active if r.get('status') == 'blocked')
    unknown = n - clear - partial - blocked

    len_clear   = sum(r['length_m'] for r in active if r.get('status') == 'clear')
    len_partial = sum(r['length_m'] for r in active if r.get('status') == 'partial')
    len_blocked = sum(r['length_m'] for r in active if r.get('status') == 'blocked')
    total_len = len_clear + len_partial + len_blocked

    avg_width = np.mean([r['avg_width_m'] for r in active]) if active else 0

    type_counts = {}
    for r in active:
        for bt in r.get('blockage_types', []):
            type_counts[bt] = type_counts.get(bt, 0) + 1
            all_blockage_types[bt] = all_blockage_types.get(bt, 0) + 1

    print(f'\n{tag}:')
    print(f'  Reaches: {n} canal-only reaches')
    print(f'  Status: clear={clear}, partial={partial}, blocked={blocked}, unknown={unknown}')
    blocked_dist = sum(r.get('blocked_distance_m', 0.0) for r in active)
    print(f'  Canal length: {total_len:.0f}m ({total_len/1000:.2f}km)')
    print(f'  Estimated blocked distance: {blocked_dist:.0f}m')
    if total_len > 0:
        print(f'    Clear:   {len_clear:.0f}m ({len_clear/total_len*100:.1f}%)')
        print(f'    Partial: {len_partial:.0f}m ({len_partial/total_len*100:.1f}%)')
        print(f'    Blocked: {len_blocked:.0f}m ({len_blocked/total_len*100:.1f}%)')
    print(f'  Avg canal width: {avg_width:.2f}m')
    if type_counts:
        print(f'  Blockage types: {type_counts}')

    total_reaches += n
    total_clear += clear
    total_partial += partial
    total_blocked += blocked
    total_length_clear += len_clear
    total_length_partial += len_partial
    total_length_blocked += len_blocked

total_length = total_length_clear + total_length_partial + total_length_blocked
print(f'\n{"="*60}')
print(f'OVERALL (canals & distributaries):')
print(f'  Total reaches: {total_reaches}')
print(f'  Total length: {total_length:.0f}m ({total_length/1000:.2f}km)')
if total_length > 0:
    print(f'    Clear:   {total_length_clear:.0f}m ({total_length_clear/total_length*100:.1f}%)')
    print(f'    Partial: {total_length_partial:.0f}m ({total_length_partial/total_length*100:.1f}%)')
    print(f'    Blocked: {total_length_blocked:.0f}m ({total_length_blocked/total_length*100:.1f}%)')
print(f'  Blockage types: {all_blockage_types}')



In [ ]:
# Cell 14 — Export blockage GeoTIFFs
# Export GeoTIFF at small resolution
for tag, cdata in canal_data.items():
    geo = cdata['geo']
    labels_s = results[tag]['labels_s']
    ds_a = results[tag]['ds_a']
    h_s, w_s = results[tag]['shape_s']
    reaches = results[tag]['reaches']

    blockage_raster = np.zeros((h_s, w_s), dtype=np.uint8)
    for reach in reaches:
        rmin_s, cmin_s, rmax_s, cmax_s = reach['bbox_s']
        cid = reach['canal_id']
        val = STATUS_VALUES.get(reach.get('status', 'unknown'), 0)
        region = labels_s[rmin_s:rmax_s, cmin_s:cmax_s]
        blockage_raster[rmin_s:rmax_s, cmin_s:cmax_s][region == cid] = val

    # Adjust transform for extra downscale
    t = geo['transform']
    out_transform = rasterio.transform.Affine(
        t.a * ds_a, t.b, t.c,
        t.d, t.e * ds_a, t.f
    )

    out_path = os.path.join(OUT_DIR, f'{tag}_blockage.tif')
    with rasterio.open(out_path, 'w', driver='GTiff',
                       height=h_s, width=w_s,
                       count=1, dtype='uint8',
                       crs=geo['crs'], transform=out_transform,
                       compress='lzw') as dst:
        dst.write(blockage_raster, 1)
        dst.update_tags(
            CLEAR='1', PARTIAL='2', BLOCKED='3', UNKNOWN='0',
            NOTE='Cleaned canal-only mask; FIC candidates excluded'
        )
    print(f'\nExported: {out_path} ({os.path.getsize(out_path)/1024:.0f} KB)')



In [ ]:
# Cell 15 — Save summary JSON
summary = {
    'scope': 'cleaned_canal_only_fic_excluded',
    'total_reaches': total_reaches,
    'total_length_m': round(total_length, 1),
    'clear_pct': round(total_length_clear / max(1, total_length) * 100, 1),
    'partial_pct': round(total_length_partial / max(1, total_length) * 100, 1),
    'blocked_pct': round(total_length_blocked / max(1, total_length) * 100, 1),
    'blockage_types': all_blockage_types,
    'per_mosaic': {}
}
for tag in canal_data:
    reaches = results[tag]['reaches']
    active = reaches
    tc = {}
    for r in active:
        for bt in r.get('blockage_types', []):
            tc[bt] = tc.get(bt, 0) + 1
    summary['per_mosaic'][tag] = {
        'reaches': len(active),
        'clear': sum(1 for r in active if r.get('status') == 'clear'),
        'partial': sum(1 for r in active if r.get('status') == 'partial'),
        'blocked': sum(1 for r in active if r.get('status') == 'blocked'),
        'blockage_types': tc,
    }

with open(os.path.join(OUT_DIR, 'blockage_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nSummary: {os.path.join(OUT_DIR, "blockage_summary.json")}')
print('GeoTIFF values: 0=background, 1=clear, 2=partial, 3=blocked')


In [ ]:
# Cell 16 — Assign global chainage
# Uses cleaned canal-only reaches from Cell 4 directly.

import matplotlib.patches as mpatches

print('Computing global chainage ...\n')

def assign_global_chainage(reaches):
    if not reaches:
        return
    centers = np.array([[r['center'][0], r['center'][1]] for r in reaches], dtype=np.float32)
    centered = centers - centers.mean(axis=0, keepdims=True)
    if len(reaches) >= 2:
        _, _, vh = np.linalg.svd(centered, full_matrices=False)
        proj = centered @ vh[0]
    else:
        proj = centers[:, 0]
    chainage = 0.0
    for seq, idx in enumerate(np.argsort(proj)):
        r = reaches[int(idx)]
        r['global_reach_idx'] = int(seq)
        r['chainage_m'] = chainage
        r['chainage_start_m'] = chainage
        chainage += r['length_m']
        r['chainage_end_m'] = chainage

for tag, res in results.items():
    assign_global_chainage(res['reaches'])
    active = [r for r in res['reaches'] if 'chainage_m' in r]
    if active:
        print(f'  {tag}: {len(active)} reaches | max chainage {max(r["chainage_end_m"] for r in active):.0f}m')



In [ ]:
# Cell 17 — Detect structural damage
print('\nDetecting structural damage ...\n')

def detect_sd(rgb_patch, canal_mask_patch):
    canal_px = canal_mask_patch > 0
    if canal_px.sum() < 20:
        return None
    gray = cv2.cvtColor(rgb_patch, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 80, 200)
    edge_density = edges[canal_px].sum() / (255.0 * canal_px.sum())
    gray_f = gray.astype(np.float32)
    lm = cv2.blur(gray_f, (7,7))
    lv = cv2.blur((gray_f - lm)**2, (7,7))
    tex_var = lv[canal_px].mean()
    gx = cv2.Sobel(gray_f, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray_f, cv2.CV_32F, 0, 1, ksize=3)
    grad = np.sqrt(gx**2 + gy**2)[canal_px].mean()
    sat = cv2.cvtColor(rgb_patch, cv2.COLOR_RGB2HSV)[:,:,1][canal_px].astype(np.float32)
    low_sat = (sat < 30).sum() / canal_px.sum()
    return {'edge_density': float(edge_density), 'tex_var': float(tex_var),
            'grad': float(grad), 'low_sat': float(low_sat)}

for tag, cdata in canal_data.items():
    fpath = cdata['fpath']
    reaches = [r for r in results[tag]['reaches'] if 'chainage_m' in r]
    with rasterio.open(fpath) as src:
        H, W = src.height, src.width
    with rasterio.open(fpath) as src:
        for reach in reaches:
            if reach.get('status') == 'blocked':
                reach['structural_damage'] = False; reach['sd_score'] = 0.0; continue
            r_min, c_min, r_max, c_max = reach['bbox']
            fr_min = r_min * DS; fc_min = c_min * DS
            fr_max = min(r_max * DS, H); fc_max = min(c_max * DS, W)
            rh, rw = fr_max - fr_min, fc_max - fc_min
            if rh < 8 or rw < 8:
                reach['structural_damage'] = False; continue
            data = src.read([1,2,3], window=Window(fc_min, fr_min, rw, rh))
            rgb_f = data.transpose(1,2,0)
            dt = DS * results[tag]['ds_a']
            ch = (rh // dt) * dt; cw = (rw // dt) * dt
            if ch < dt or cw < dt:
                reach['structural_damage'] = False; continue
            rgb_sm = rgb_f[:ch,:cw].reshape(ch//dt, dt, cw//dt, dt, 3).mean(axis=(1,3)).astype(np.uint8)
            rs, cs, re, ce = reach['bbox_s']
            ls = results[tag]['labels_s']
            cid = reach['canal_id']
            cp = (ls[rs:rs+rgb_sm.shape[0], cs:cs+rgb_sm.shape[1]] == cid).astype(np.uint8)
            if cp.shape != rgb_sm.shape[:2]:
                mh = min(cp.shape[0], rgb_sm.shape[0]); mw = min(cp.shape[1], rgb_sm.shape[1])
                cp = cp[:mh,:mw]; rgb_sm = rgb_sm[:mh,:mw]
            sd = detect_sd(rgb_sm, cp)
            if sd is None:
                reach['structural_damage'] = False; continue
            sc = 0.0
            if sd['edge_density'] > 0.35: sc += 2.0
            elif sd['edge_density'] > 0.25: sc += 1.0
            if sd['grad'] > 80: sc += 2.0
            elif sd['grad'] > 60: sc += 1.0
            if sd['low_sat'] > 0.7: sc += 2.0
            elif sd['low_sat'] > 0.5: sc += 1.0
            if sd['tex_var'] > 1500: sc += 1.0
            reach['structural_damage'] = sc >= 4.0
            reach['sd_score'] = sc
    sd_n = sum(1 for r in reaches if r.get('structural_damage'))
    print(f'  {tag}: {sd_n}/{len(reaches)} structural damage')



In [ ]:
# Cell 18 — Bounding-box chainage maps
print('\nGenerating bounding box maps ...\n')
VIS_MAX = 2500

def cstretch(rgb, valid):
    out = np.zeros_like(rgb, dtype=np.uint8)
    for c in range(3):
        ch = rgb[:,:,c].astype(np.float64)
        vp = ch[valid]
        if vp.size < 100: continue
        lo, hi = np.percentile(vp, [2, 98])
        if hi - lo < 5: continue
        out[:,:,c] = np.clip((ch - lo)/(hi-lo)*255, 0, 255).astype(np.uint8)
    return out

for tag, cdata in canal_data.items():
    fpath = cdata['fpath']
    canal_bin = cdata['binary']
    h_full, w_full = canal_bin.shape
    vs = max(1, max(h_full, w_full) // VIS_MAX)
    vh, vw = h_full // vs, w_full // vs
    with rasterio.open(fpath) as src:
        thumb = src.read([1,2,3,4], out_shape=(4, vh, vw))
    rgb_vis = cstretch(thumb[:3].transpose(1,2,0), thumb[3] == 255)
    del thumb

    overlay = rgb_vis.copy()
    reaches = [r for r in results[tag]['reaches'] if 'chainage_m' in r]
    for reach in reaches:
        r_min, c_min, r_max, c_max = reach['bbox']
        y1, x1 = max(0, int(r_min/vs)), max(0, int(c_min/vs))
        y2, x2 = min(vh-1, int(r_max/vs)), min(vw-1, int(c_max/vs))
        status = reach.get('status', 'unknown')
        sd = reach.get('structural_damage', False)
        if sd:
            color = (255, 0, 255); lp = 'SD'
        elif status == 'blocked':
            color = (220, 20, 20); lp = 'VE'
        elif status == 'partial':
            color = (255, 165, 0); lp = 'VE'
        elif status == 'clear':
            color = (0, 200, 0); lp = ''
        else:
            color = (128, 128, 128); lp = ''
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
        ch0 = reach.get('chainage_start_m', reach.get('chainage_m', 0))
        ch1 = reach.get('chainage_end_m', ch0 + reach.get('length_m', 0))
        bd = reach.get('blocked_distance_m', 0.0)
        label = f'{lp} {ch0:.0f}-{ch1:.0f}m' if lp else f'{ch0:.0f}-{ch1:.0f}m'
        if bd > 0:
            label += f' B{bd:.0f}m'
        fs = 0.32; th = 1
        (tw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, fs, th)
        tx, ty = x1, max(y1, lh + 4)
        cv2.rectangle(overlay, (tx, ty-lh-3), (tx+tw+2, ty+1), (0,0,0), -1)
        cv2.putText(overlay, label, (tx+1, ty-1), cv2.FONT_HERSHEY_SIMPLEX, fs, (255,255,255), th)

    fig, ax = plt.subplots(figsize=(12, int(12 * vh/vw) + 1))
    ax.imshow(overlay)
    sc = {}
    for r in reaches:
        s = 'SD' if r.get('structural_damage') else r.get('status','?')
        sc[s] = sc.get(s,0) + 1
    ax.set_title(f'{tag} — Canal-only Blockage Map with Chainage\nCounts: {sc}', fontsize=11)
    ax.axis('off')
    ax.legend(handles=[
        mpatches.Patch(color=(220/255,20/255,20/255), label='VE: Blocked'),
        mpatches.Patch(color=(255/255,165/255,0), label='VE: Partial'),
        mpatches.Patch(color=(255/255,0,255/255), label='SD: Structural Damage'),
        mpatches.Patch(color=(0,200/255,0), label='Clear'),
    ], loc='lower right', fontsize=9)

    # Count SD and VE
    sd_count = sum(1 for r in reaches if r.get('structural_damage'))
    ve_blocked = sum(1 for r in reaches if r.get('status') == 'blocked')
    ve_partial = sum(1 for r in reaches if r.get('status') == 'partial')

    # Add text box with counts
    count_text = f'SD: {sd_count}\nVE (Blocked): {ve_blocked}\nVE (Partial): {ve_partial}'
    ax.text(0.02, 0.98, count_text,
            transform=ax.transAxes,
            fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f'{tag}_bbox_chainage_map.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  {tag}: {len(reaches)} reaches drawn | Saved {tag}_bbox_chainage_map.png')
    del overlay, rgb_vis; gc.collect()



In [ ]:
# Cell 19 — Close-up patches of top reaches
print('\nClose-up patches ...\n')

def cstretch_patch(rgb):
    out = np.zeros_like(rgb, dtype=np.uint8)
    for c in range(3):
        ch = rgb[:,:,c].astype(np.float64)
        v = ch[ch > 0]
        if v.size < 10: continue
        lo, hi = np.percentile(v, [2, 98])
        if hi - lo < 5: continue
        out[:,:,c] = np.clip((ch-lo)/(hi-lo)*255, 0, 255).astype(np.uint8)
    return out

all_r = []
for tag in canal_data:
    for r in results[tag]['reaches']:
        if 'status' in r and r['status'] != 'unknown':
            all_r.append((tag, r))
all_r.sort(key=lambda x: (x[1].get('blockage_score', 0), x[1].get('blocked_distance_m', 0)), reverse=True)

show = all_r[:12]
N = len(show)
cols = 4; rows = max(1, (N + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
if rows == 1: axes = axes[np.newaxis, :]

open_f = {}
for i, (tag, reach) in enumerate(show):
    ax = axes[i//cols, i%cols]
    r_min, c_min, r_max, c_max = reach['bbox']
    fpath = canal_data[tag]['fpath']
    if tag not in open_f: open_f[tag] = rasterio.open(fpath)
    src = open_f[tag]
    fr_min, fc_min = r_min*DS, c_min*DS
    fr_max = min(r_max*DS, src.height); fc_max = min(c_max*DS, src.width)
    rh, rw = fr_max-fr_min, fc_max-fc_min
    if rh < 4 or rw < 4: ax.axis('off'); continue
    data = src.read([1,2,3], window=Window(fc_min, fr_min, rw, rh))
    rgb = data.transpose(1,2,0)
    if max(rh,rw) > 400:
        sc2 = 400/max(rh,rw)
        rgb = cv2.resize(rgb, (max(1, int(rw*sc2)), max(1, int(rh*sc2))), interpolation=cv2.INTER_AREA)
    rgb = cstretch_patch(rgb)
    rs, cs, re, ce = reach['bbox_s']
    ls = results[tag]['labels_s']
    cid = reach['canal_id']
    cm_s = (ls[rs:re, cs:ce] == cid).astype(np.uint8)
    cm_r = cv2.resize(cm_s, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
    status = reach.get('status', 'unknown')
    sd = reach.get('structural_damage', False)
    if sd: col = np.array([255,0,255])
    elif status == 'blocked': col = np.array([220,20,20])
    elif status == 'partial': col = np.array([255,165,0])
    elif status == 'clear': col = np.array([0,200,0])
    else: col = np.array([128,128,128])
    ov = rgb.copy()
    mb = cm_r > 0
    ov[mb] = (ov[mb]*0.45 + col*0.55).astype(np.uint8)
    cnts, _ = cv2.findContours(cm_r, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(ov, cnts, -1, (255,255,255), 1)
    cv2.rectangle(ov, (0,0), (ov.shape[1]-1, ov.shape[0]-1), tuple(col.tolist()), 3)
    ax.imshow(ov)
    feats = reach.get('features', {}) or {}
    ch0 = reach.get('chainage_start_m', reach.get('chainage_m', 0))
    ch1 = reach.get('chainage_end_m', ch0 + reach.get('length_m', 0))
    bd = reach.get('blocked_distance_m', 0)
    lbl = 'SD' if sd else status.upper()
    t = f'{tag} | {lbl} | Ch:{ch0:.0f}-{ch1:.0f}m\nW:{reach["avg_width_m"]:.1f}m L:{reach["length_m"]:.0f}m Blk:{bd:.0f}m'
    if feats: t += f'\nveg={feats.get("veg_frac",0):.0%} water={feats.get("water_frac",0):.0%}'
    ax.set_title(t, fontsize=7.5); ax.axis('off')

for i in range(N, rows*cols): axes[i//cols, i%cols].axis('off')
for f in open_f.values(): f.close()
plt.suptitle('Close-up Canal-only Reaches — Sorted by Blockage Severity', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'closeup_reaches.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: closeup_reaches.png')



In [ ]:
# Cell 20 — Final reach summary table
print(f'\n{"="*125}')
print(f'{"Tag":<12} {"Idx":>4} {"Status":<10} {"Type":<5} {"ChStart":>8} {"ChEnd":>8} {"W(m)":>5} {"L(m)":>6} {"Blk(m)":>7} {"Blk%":>5} {"Veg%":>5} {"Wat%":>5}')
print(f'{"="*125}')
for tag in canal_data:
    reaches = sorted([r for r in results[tag]['reaches'] if 'chainage_m' in r], key=lambda r: r.get('chainage_m', 0))
    for r in reaches:
        status = r.get('status', '?')
        sd = r.get('structural_damage', False)
        itype = 'SD' if sd else ('VE' if status in ['blocked','partial'] else '--')
        bd = r.get('blocked_distance_m', 0)
        bf = r.get('blocked_fraction', 0) * 100
        feats = r.get('features') or {}
        print(f'{tag:<12} {r.get("global_reach_idx",0):>4} {status:<10} {itype:<5} '
              f'{r.get("chainage_start_m",0):>8.1f} {r.get("chainage_end_m",0):>8.1f} '
              f'{r["avg_width_m"]:>5.1f} {r["length_m"]:>6.0f} {bd:>7.1f} {bf:>5.1f} '
              f'{feats.get("veg_frac",0)*100:>5.1f} {feats.get("water_frac",0)*100:>5.1f}')
print(f'{"="*125}')
print('VE=Vegetation Encroachment  SD=Structural Damage  Blk=blocked centerline length estimate')
